In [1]:
#%pip install -q llama-index llama-index-llms-anthropic llama-index-embeddings-cohere llama-index-readers-file llama-index-readers-web llama-index-readers-wikipedia wikipedia python-dotenv

# Importar Librerias

In [2]:
import os, json, pathlib, logging, warnings
warnings.filterwarnings("ignore", message="IProgress not found")
from dotenv import load_dotenv
from llama_index.core import SimpleDirectoryReader, Document, VectorStoreIndex, StorageContext, Settings, load_index_from_storage
from llama_index.core.node_parser import SimpleNodeParser, TokenTextSplitter
from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter
from llama_index.llms.anthropic import Anthropic
from llama_index.embeddings.cohere import CohereEmbedding
from llama_index.readers.file import PDFReader

# Suprimir logs informativos de red (Cohere/httpx)

In [3]:
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("cohere").setLevel(logging.WARNING)

# Cargar credenciales desde .env

In [4]:
CARPETA_CONN    = r"C:\Users\milu_\ricardo palma ia\Proyecto NutriAgente\Conexiones"
CARPETA_BDC     = r"C:\Users\milu_\ricardo palma ia\Proyecto NutriAgente\Base_Conocimiento"
CARPETA_IDX     = r"C:\Users\milu_\ricardo palma ia\Proyecto NutriAgente\Base_Conocimiento\indice_platos"
CARPETA_FUENTES = r"C:\Users\milu_\ricardo palma ia\Proyecto NutriAgente\Fuentes"
 
load_dotenv(os.path.join(CARPETA_CONN, ".env"))
 
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
COHERE_API_KEY    = os.getenv("COHERE_API_KEY")
 
print("Credenciales cargadas ✓")

Credenciales cargadas ✓


# LLM: Anthropic Claude Haiku

In [5]:
llm = Anthropic(
    model="claude-haiku-4-5-20251001",
    api_key=ANTHROPIC_API_KEY,
    max_tokens=1024,
)
Settings.llm = llm
print("LLM: Claude Haiku ✓")

LLM: Claude Haiku ✓


# Embeddings: Cohere multilingual

In [6]:
embed_model = CohereEmbedding(
    cohere_api_key=COHERE_API_KEY,
    model_name="embed-multilingual-v3.0",
    input_type="search_document",
)
Settings.embed_model = embed_model
print("Embeddings: Cohere embed-multilingual-v3.0 ✓")

Embeddings: Cohere embed-multilingual-v3.0 ✓


# LOADER 1: JSONReader → platos_kb.json

In [7]:
# 512 platos extraídos y limpios por Ingeniería de Datos (Archivo 1)
# Campos: nombre, insumo, rol, turno, guarniciones, fuente
 
platos_path   = pathlib.Path(CARPETA_BDC) / "platos_kb.json"
docs_platos   = SimpleDirectoryReader(input_files=[str(platos_path)]).load_data()
 
print(f"\nLOADER 1 — platos_kb.json")
print(f"  Documentos: {len(docs_platos)}")
print(f"  Ejemplo: {docs_platos[0]}")


LOADER 1 — platos_kb.json
  Documentos: 1
  Ejemplo: Doc ID: 6d554d72-e27d-46a9-956f-d086f905d211
Text: [    {      "nombre": "Ceviche de Pollo",      "insumo":
"Pollo",      "rol": "Fondo",      "turno": "Almuerzo",
"guarniciones": [        "Frejol",        "Camote",        "Alverja",
"Yuca",        "Lenteja"      ],      "fuente": "PLATOS PLANTA.xlsx"
},    {      "nombre": "Escabeche de Pollo",      "insumo": "Pollo",
"rol":...


# LOADER 2: SimpleDirectoryReader → reglas_semaforo.json

In [8]:
# 6 dimensiones nutricionales con criterios Verde / Ámbar / Rojo
# Fuente original: SEMAFORO DE LA NUTRICION.xlsx (procesado en Archivo 1)
semaforo_path = pathlib.Path(CARPETA_BDC) / "reglas_semaforo.json"
docs_semaforo = SimpleDirectoryReader(input_files=[str(semaforo_path)]).load_data()
 
print(f"\nLOADER 2 — reglas_semaforo.json")
print(f"  Documentos: {len(docs_semaforo)}")
print(f"  Ejemplo: {docs_semaforo[0]}")


LOADER 2 — reglas_semaforo.json
  Documentos: 1
  Ejemplo: Doc ID: 0a24952b-7f91-4e04-abcf-a60550fee4ac
Text: {    "Energía": {      "verde": "Si la preparación contiene al
menos 50% de verduras",      "ambar": "Si la preparación contiene un
tuberculo, guiso, o menestra",      "rojo": "Si la preparación
contiene fritura, apanado, plato a base de arroz o plato a base de
fideos"    },    "Carbohidratos": {      "verde": "Si la preparación
contiene solo 1 ...


# LOADER 3: PDFReader → Guías alimentarias MINSA

In [9]:
# Archivo 1 solo extrae nombres de platos de este PDF
# LlamaIndex indexa el texto completo para responder preguntas nutricionales
pdf_reader   = PDFReader()
pdf_guias    = pathlib.Path(CARPETA_FUENTES) / "Guías_alimentarias para la poblacion peruana.pdf"
docs_guias   = pdf_reader.load_data(file=pdf_guias)
 
print(f"\nLOADER 3 — PDFReader (Guías alimentarias MINSA)")
print(f"  Páginas cargadas: {len(docs_guias)}")
print(f"  Ejemplo: {docs_guias[0].text[:150]}")


LOADER 3 — PDFReader (Guías alimentarias MINSA)
  Páginas cargadas: 60
  Ejemplo: 


# LOADER 4: PDFReader → CENAN-0013 MINSA

In [10]:
# Norma técnica de composición nutricional de recetas peruanas
pdf_cenan    = pathlib.Path(CARPETA_FUENTES) / "CENAN-0013.pdf"
docs_cenan   = pdf_reader.load_data(file=pdf_cenan)
 
print(f"\nLOADER 4 — PDFReader (CENAN-0013 MINSA)")
print(f"  Páginas cargadas: {len(docs_cenan)}")
print(f"  Ejemplo: {docs_cenan[0].text[:150]}")


LOADER 4 — PDFReader (CENAN-0013 MINSA)
  Páginas cargadas: 206
  Ejemplo: ...nutritivo, económico y saludable
RecetarioRecetarioRecetario...nutritivo, económico y saludable...nutritivo, económico y saludable
Lima - Perú
2011


# METADATA — Enriquecer platos con campos filtrables

In [11]:
# Cada plato lleva metadata: rol, insumo, turno, fuente
# Esto permite filtrar búsquedas (ej: solo Fondos de Pollo en Almuerzo)
with open(platos_path, encoding="utf-8") as f:
    platos_raw = json.load(f)
 
docs_platos_meta = []
for p in platos_raw:
    texto = (
        f"Plato: {p['nombre']}\n"
        f"Insumo: {p.get('insumo', '')}\n"
        f"Rol: {p.get('rol', '')}\n"
        f"Turno: {p.get('turno', '')}\n"
        f"Guarniciones: {', '.join(p.get('guarniciones', []))}"
    )
    doc = Document(
        text=texto,
        metadata={
            "nombre": p["nombre"],
            "insumo": p.get("insumo", ""),
            "rol":    p.get("rol", ""),
            "turno":  p.get("turno", ""),
            "fuente": p.get("fuente", ""),
        }
    )
    docs_platos_meta.append(doc)
 
print(f"\nPlatos con metadata: {len(docs_platos_meta)}")
print(f"Metadata ejemplo: {docs_platos_meta[0].metadata}")


Platos con metadata: 512
Metadata ejemplo: {'nombre': 'Ceviche de Pollo', 'insumo': 'Pollo', 'rol': 'Fondo', 'turno': 'Almuerzo', 'fuente': 'PLATOS PLANTA.xlsx'}


# FRAGMENTACIÓN

In [12]:
# Platos: 1 plato = 1 nodo (ya son unidades pequeñas, no fragmentar)
parser_platos  = SimpleNodeParser.from_defaults(chunk_size=512, chunk_overlap=0)
nodes_platos   = parser_platos.get_nodes_from_documents(docs_platos_meta)
 
# Textos largos (PDFs MINSA + reglas): fragmentar por tokens
parser_textos  = TokenTextSplitter(chunk_size=512, chunk_overlap=50)
nodes_textos   = parser_textos.get_nodes_from_documents(
    docs_semaforo + docs_guias + docs_cenan
)
 
all_nodes = nodes_platos + nodes_textos
print(f"\nFragmentación:")
print(f"  Nodos platos  : {len(nodes_platos)}")
print(f"  Nodos textos  : {len(nodes_textos)}")
print(f"  Total nodos   : {len(all_nodes)}")


Fragmentación:
  Nodos platos  : 512
  Nodos textos  : 317
  Total nodos   : 829


# VECTORIZACIÓN Y ALMACENAMIENTO — SimpleVectorStore (sin costo)

In [13]:
# Guarda en disco — reemplaza Azure AI Search ($7/día)
import time
pathlib.Path(CARPETA_IDX).mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 200
print(f"Vectorizando {len(all_nodes)} nodos en tandas de {BATCH_SIZE}...")

index = None
for i in range(0, len(all_nodes), BATCH_SIZE):
    tanda = all_nodes[i : i + BATCH_SIZE]
    print(f"  Tanda {i//BATCH_SIZE + 1}: nodos {i+1}–{min(i+BATCH_SIZE, len(all_nodes))}...")
    if index is None:
        index = VectorStoreIndex(nodes=tanda, embed_model=embed_model, show_progress=True)
    else:
        index.insert_nodes(tanda)
    if i + BATCH_SIZE < len(all_nodes):
        print("  Esperando 65 segundos (rate limit Cohere trial)...")
        time.sleep(65)

index.storage_context.persist(persist_dir=CARPETA_IDX)
print(f"Índice guardado en: {CARPETA_IDX} ✓")

Vectorizando 829 nodos en tandas de 200...
  Tanda 1: nodos 1–200...


Generating embeddings: 100%|██████████| 200/200 [00:09<00:00, 22.13it/s]


  Esperando 65 segundos (rate limit Cohere trial)...
  Tanda 2: nodos 201–400...
  Esperando 65 segundos (rate limit Cohere trial)...
  Tanda 3: nodos 401–600...
  Esperando 65 segundos (rate limit Cohere trial)...
  Tanda 4: nodos 601–800...
  Esperando 65 segundos (rate limit Cohere trial)...
  Tanda 5: nodos 801–829...
Índice guardado en: C:\Users\milu_\ricardo palma ia\Proyecto NutriAgente\Base_Conocimiento\indice_platos ✓


# Pruebas

In [14]:
embed_query = CohereEmbedding(
    cohere_api_key=COHERE_API_KEY,
    model_name="embed-multilingual-v3.0",
    input_type="search_query",
)
 
def buscar_con_filtros(consulta: str, rol: str = None, insumo: str = None,
                       turno: str = None, top: int = 5):
    """Búsqueda semántica con filtros opcionales de metadata."""
    filtros = []
    if rol:    filtros.append(ExactMatchFilter(key="rol",    value=rol))
    if insumo: filtros.append(ExactMatchFilter(key="insumo", value=insumo))
    if turno:  filtros.append(ExactMatchFilter(key="turno",  value=turno))
 
    retriever = index.as_retriever(
        similarity_top_k=top,
        embed_model=embed_query,
        filters=MetadataFilters(filters=filtros) if filtros else None,
    )
    nodos = retriever.retrieve(consulta)
    print(f"\nConsulta: '{consulta}' | rol={rol} | insumo={insumo} | turno={turno}")
    print(f"Resultados: {len(nodos)}")
    for n in nodos:
        m = n.metadata
        print(f"  [{round(n.score,3)}] {m.get('nombre','')} | {m.get('rol','')} | {m.get('insumo','')} | {m.get('turno','')}")
    return nodos
 
# Prueba 1: búsqueda libre (puede traer nodos de PDFs — metadata vacía es normal)
print("\n=== PRUEBA 1: Búsqueda libre ===")
print("  Nota: sin filtros puede retornar fragmentos de PDFs (sin nombre/rol/turno)")
buscar_con_filtros("algo con pollo")
 
# Prueba 2: filtro por rol + insumo
print("\n=== PRUEBA 2: Fondos de Pollo ===")
buscar_con_filtros("pollo", rol="Fondo", insumo="Pollo")
 
# Prueba 3: filtro por turno
print("\n=== PRUEBA 3: Sopas para Amanecida ===")
buscar_con_filtros("sopa caliente", rol="Sopa", turno="Amanecida")
 
# Prueba 4: búsqueda semántica en reglas nutricionales (PDFs MINSA + semáforo)
# La síntesis con LLM ocurre en Archivo 3 (Agente) — aquí solo se valida la recuperación
print("\n=== PRUEBA 4: Búsqueda en reglas nutricionales ===")
nodos_p4 = buscar_con_filtros("grasas plato rojo semáforo", top=3)
for n in nodos_p4:
    print(f"\n  Fuente: {n.metadata.get('file_name', 'PDF/reglas')}")
    print(f"  {n.text[:250]}")
 
print("\n=== PIPELINE COMPLETADO ===")
print(f"  Loaders    : 4 (SimpleDirectoryReaderx2, PDFReaderx2)")
print(f"  Nodos      : {len(all_nodes)}")
print(f"  Embeddings : Cohere embed-multilingual-v3.0")
print(f"  Storage    : SimpleVectorStore → {CARPETA_IDX}")
 
 


=== PRUEBA 1: Búsqueda libre ===
  Nota: sin filtros puede retornar fragmentos de PDFs (sin nombre/rol/turno)

Consulta: 'algo con pollo' | rol=None | insumo=None | turno=None
Resultados: 5
  [0.636]  |  |  | 
  [0.629]  |  |  | 
  [0.627]  |  |  | 
  [0.626]  |  |  | 
  [0.619]  |  |  | 

=== PRUEBA 2: Fondos de Pollo ===

Consulta: 'pollo' | rol=Fondo | insumo=Pollo | turno=None
Resultados: 5
  [0.57] Pollo A Lo Criollo | Fondo | Pollo | Almuerzo
  [0.569] Pollo al Horno | Fondo | Pollo | Almuerzo
  [0.567] Pollo a la Olla | Fondo | Pollo | Almuerzo
  [0.566] Pollo Enrrollado c/ Salsa Blanca | Fondo | Pollo | Almuerzo
  [0.566] Pollo al Sillao | Fondo | Pollo | Almuerzo

=== PRUEBA 3: Sopas para Amanecida ===

Consulta: 'sopa caliente' | rol=Sopa | insumo=None | turno=Amanecida
Resultados: 2
  [0.583] Shambar Presa | Sopa | Caldos Fuertes | Amanecida
  [0.557] Aguadito con Pollo Presa | Sopa | Caldos Fuertes | Amanecida

=== PRUEBA 4: Búsqueda en reglas nutricionales ===

Consulta: 